In [34]:
def initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose):
    num_evals_init = num_evals
    init_range = hi - lo
    if is_pos:
        mse, z, ghat = eval_fn(hi)
        while mse < epsilon_squared and num_evals > 0:
            hi += init_range
            mse, z, ghat = eval_fn(hi)
            num_evals -= 1
        bound_dict = {'lo':mse_z_ghat_0, 'hi':(mse, z, ghat)}
    else:
        mse, z, ghat = eval_fn(lo)
        while mse > epsilon_squared and num_evals > 0: # remember epsilon_squared will be negative in this case
            lo -= init_range
            mse, z, ghat = eval_fn(lo)
            num_evals -= 1
        bound_dict = {'lo':(mse, z, ghat), 'hi':mse_z_ghat_0}
    if num_evals == 0:
        raise ValueError('Exceeded number of allowable evaluations during initialization of search bounds.')
    elif verbose:
        print(f'Initial bounds: ({lo}, {hi})')
        print(f'{num_evals}/{num_evals_init} evaluations remaining after initialization.')
    return lo, hi, bound_dict, num_evals

def bisection_search(lo, hi , eval_fn, num_evals, epsilon_squared, mse_z_ghat_0, verbose, tol=0):
    is_pos = hi > 0
    lo, hi, bound_dict, num_evals = initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose)
    while lo < hi - tol and num_evals > 0:
        mid = (lo + hi) / 2
        mse, z, ghat = eval_fn(mid)
        if verbose:
            print(f'alpha:{mid}, mse:{mse}, lo:{lo}, hi:{hi}')
        if mse < epsilon_squared:
            lo = mid
            bound_dict['lo'] = (mse, z, ghat)
        elif mse > epsilon_squared:
            hi = mid
            bound_dict['hi'] = (mse, z, ghat)
        else:
            return (mid, mse, z, ghat)
        num_evals -= 1
    return (hi,) + bound_dict['hi'] if is_pos else (lo,) + bound_dict['lo']

            

# def optimize_alpha(vanilla_dy_dx, zo_dy_dx, net, criterion, method, gt_data, 
#                    label_pred, num_attack_iterations, num_dummy, imidx_list,
#                    num_alpha_search_iterations, epsilon_squared):
#      inv_attack_closure = lambda ghat: inv_attack(ghat, net, criterion, method, gt_data, label_pred, 
#                                        num_attack_iterations, None, num_dummy, None, 
#                                        imidx_list, None, False)
def inv_attack_closure(alpha):
    return alpha * alpha, 'z', 'ghat'
def get_bisection_search_eval_fn(sign):
    if sign == 'pos':
        def bisection_search_eval_fn(alpha):
            return inv_attack_closure(alpha)
    elif sign == 'neg':
        def bisection_search_eval_fn(alpha):
            mse, z, ghat = inv_attack_closure(alpha)
            return -mse, z, ghat
    else:
        raise ValueError('sign must be either `pos` or `neg')
    return bisection_search_eval_fn

            
epsilon_squared = 0.1
num_alpha_search_iterations = 10
mse_0, z_0, ghat_0 = inv_attack_closure(0)
if mse_0 >= epsilon_squared:
    # if the vanilla gradient (alpha=0) is already larger than the error tol, then we are satisfying the constraint and can't reduce alpha any further. return 
    print( 0, ghat_0)
alpha_star_pos, mse_star_pos, zhat_pos, ghat_alpha_star_pos = bisection_search(0, 1, get_bisection_search_eval_fn('pos'), num_alpha_search_iterations, epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)
alpha_star_neg, mse_star_neg, zhat_neg, ghat_alpha_star_neg = bisection_search(-1, 0, get_bisection_search_eval_fn('neg'), num_alpha_search_iterations, -epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)


Initial bounds: (0, 1)
10/10 evaluations remaining after initialization.
alpha:0.5, mse:0.25, lo:0, hi:1
alpha:0.25, mse:0.0625, lo:0, hi:0.5
alpha:0.375, mse:0.140625, lo:0.25, hi:0.5
alpha:0.3125, mse:0.09765625, lo:0.25, hi:0.375
alpha:0.34375, mse:0.1181640625, lo:0.3125, hi:0.375
alpha:0.328125, mse:0.107666015625, lo:0.3125, hi:0.34375
alpha:0.3203125, mse:0.10260009765625, lo:0.3125, hi:0.328125
alpha:0.31640625, mse:0.1001129150390625, lo:0.3125, hi:0.3203125
alpha:0.314453125, mse:0.09888076782226562, lo:0.3125, hi:0.31640625
alpha:0.3154296875, mse:0.09949588775634766, lo:0.314453125, hi:0.31640625
Initial bounds: (-1, 0)
10/10 evaluations remaining after initialization.
alpha:-0.5, mse:-0.25, lo:-1, hi:0
alpha:-0.25, mse:-0.0625, lo:-0.5, hi:0
alpha:-0.375, mse:-0.140625, lo:-0.5, hi:-0.25
alpha:-0.3125, mse:-0.09765625, lo:-0.375, hi:-0.25
alpha:-0.34375, mse:-0.1181640625, lo:-0.375, hi:-0.3125
alpha:-0.328125, mse:-0.107666015625, lo:-0.34375, hi:-0.3125
alpha:-0.3203125,

In [35]:
alpha_star_pos

0.31640625

In [36]:
alpha_star_neg

-0.31640625

# Assume the Objective is Noisy and Decide alpha via stochastic sampling and shrinking confidence interval

In [ ]:
import scipy.stats as stats
import numpy as np
from matplotlib import pyplot as plt

def initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose):
    num_evals_init = num_evals
    init_range = hi - lo
    if is_pos:
        mse, z, ghat = eval_fn(hi)
        num_evals -= 1
        while mse < epsilon_squared and num_evals > 0:
            hi += init_range
            mse, z, ghat = eval_fn(hi)
            num_evals -= 1
        bound_dict = {'lo':mse_z_ghat_0, 'hi':(mse, z, ghat)}
    else:
        mse, z, ghat = eval_fn(lo)
        num_evals -= 1
        while mse > epsilon_squared and num_evals > 0: # remember epsilon_squared will be negative in this case
            lo -= init_range
            mse, z, ghat = eval_fn(lo)
            num_evals -= 1
        bound_dict = {'lo':(mse, z, ghat), 'hi':mse_z_ghat_0}
    if num_evals <= 0:
        raise ValueError('Exceeded number of allowable evaluations during initialization of search bounds.')
    elif verbose:
        print(f'Initial bounds: ({lo}, {hi})')
        print(f'{num_evals}/{num_evals_init} evaluations remaining after initialization.')
    return lo, hi, bound_dict, num_evals


def get_v_statistic(mses, epsilon_squared):
    return int(sum(mses < epsilon_squared))

def get_ci_bounds(mses, epsilon_squared, delta):
    n = len(mses)
    if n == 0:
        raise ValueError("mses must be non-empty")
    v = get_v_statistic(np.array(mses), epsilon_squared)
    if v == 0:
        theta_l = 0
        theta_u = stats.beta.ppf(1 - delta/2, 1, n) 
    elif v == n:
        theta_l = stats.beta.ppf(delta/2, n, 1)       # Beta(n, 1)
        theta_u = 1.0
    else:
        theta_l = stats.beta.ppf(delta/2, v, n-v+1)
        theta_u = stats.beta.ppf(1-delta/2, v+1, n-v)
    return theta_l, theta_u

def get_ci_protection_output(alpha, epsilon_squared, eval_fn, num_evals, num_init_samples, tau=0.1, delta=0.1, tol=0.0000001, doplot=False):
    if num_evals <= 0:
        return 'inconclusive', num_evals
    num_init_samples = min(num_evals, num_init_samples)
    mses = [eval_fn(alpha)[0] for _ in range(num_init_samples)]
    num_evals -= num_init_samples
    theta_l, theta_u = get_ci_bounds(mses, epsilon_squared, delta)
    if tau > theta_u: return 'safe', num_evals
    if tau < theta_l: return 'unsafe', num_evals
    while True:
        if theta_u - theta_l < tol: return 'boundary', num_evals
        if num_evals <= 0: return 'inconclusive', num_evals
        new_mse = eval_fn(alpha)[0]
        num_evals -= 1
        mses.append(new_mse)
        theta_l, theta_u = get_ci_bounds(mses, epsilon_squared, delta)
        if tau > theta_u:
            return 'safe', num_evals
        elif tau < theta_l:
            return 'unsafe', num_evals
    
def custom_bisection_search(lo, hi , eval_fn, num_evals, epsilon_squared, mse_z_ghat_0, verbose, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol, tol=0):
    is_pos = hi > 0
    lo, hi, _, num_evals = initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose)
    while lo < hi - tol and num_evals > 0:
        mid = (lo + hi) / 2
        is_protected, num_evals = get_ci_protection_output(mid, epsilon_squared, eval_fn, num_evals, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol)
        if verbose:
            print(f'Evals remaning: {num_evals}, alpha:{mid}, protection_status:{is_protected}, lo:{lo}, hi:{hi}')
        if is_protected == 'unsafe':
            lo = mid
        elif is_protected == 'safe':
            hi = mid
        elif is_protected == 'boundary':
            return mid
        else: # is_protected == 'inconclusive'
            assert num_evals <= 0
            return hi if is_pos else lo
    return hi if is_pos else lo

            

# def optimize_alpha(vanilla_dy_dx, zo_dy_dx, net, criterion, method, gt_data, 
#                    label_pred, num_attack_iterations, num_dummy, imidx_list,
#                    num_alpha_search_iterations, epsilon_squared):
#      inv_attack_closure = lambda ghat: inv_attack(ghat, net, criterion, method, gt_data, label_pred, 
#                                        num_attack_iterations, None, num_dummy, None, 
#                                        imidx_list, None, False)

noise_var = 0.01
def inv_attack_closure(alpha):
    return alpha * alpha + stats.norm(loc=0, scale=np.sqrt(noise_var)).rvs(), 'z', 'ghat'
def get_bisection_search_eval_fn(sign):
    if sign == 'pos':
        def bisection_search_eval_fn(alpha):
            return inv_attack_closure(alpha)
    elif sign == 'neg':
        def bisection_search_eval_fn(alpha):
            mse, z, ghat = inv_attack_closure(alpha)
            return -mse, z, ghat
    else:
        raise ValueError('sign must be either `pos` or `neg')
    return bisection_search_eval_fn

            
epsilon_squared = 0.1
num_alpha_search_iterations = 1000
ci_protection_num_init_samples = 10
ci_protection_delta = 0.1
ci_protection_tau = 0.1
ci_protection_tol = 0.0000001
mse_0, z_0, ghat_0 = inv_attack_closure(0)
if mse_0 >= epsilon_squared:
    # if the vanilla gradient (alpha=0) is already larger than the error tol, then we are satisfying the constraint and can't reduce alpha any further. return 
    print( 0, ghat_0)
alpha_star_pos = custom_bisection_search(0, 1, get_bisection_search_eval_fn('pos'), num_alpha_search_iterations, epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol)
alpha_star_neg = custom_bisection_search(-1, 0, get_bisection_search_eval_fn('neg'), num_alpha_search_iterations, -epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol)


Initial bounds: (0, 1)
999/1000 evaluations remaining after initialization.
Evals remaning: 970, alpha:0.5, protection_status:safe, lo:0, hi:1
Evals remaning: 960, alpha:0.25, protection_status:unsafe, lo:0, hi:0.5
Evals remaning: 949, alpha:0.375, protection_status:unsafe, lo:0.25, hi:0.5
Evals remaning: 902, alpha:0.4375, protection_status:unsafe, lo:0.375, hi:0.5
Evals remaning: 0, alpha:0.46875, protection_status:inconclusive, lo:0.4375, hi:0.5
Initial bounds: (-1, 0)
999/1000 evaluations remaining after initialization.
Evals remaning: 989, alpha:-0.5, protection_status:unsafe, lo:-1, hi:0
Evals remaning: 979, alpha:-0.25, protection_status:unsafe, lo:-0.5, hi:0
Evals remaning: 969, alpha:-0.125, protection_status:unsafe, lo:-0.25, hi:0
Evals remaning: 790, alpha:-0.0625, protection_status:unsafe, lo:-0.125, hi:0
Evals remaning: 624, alpha:-0.03125, protection_status:unsafe, lo:-0.0625, hi:0
Evals remaning: 506, alpha:-0.015625, protection_status:unsafe, lo:-0.03125, hi:0
Evals rem

In [32]:
alpha_star_pos

0.5

In [33]:
alpha_star_neg

-0.001953125